In [1]:
import os
import polars as pl
LANDING_DIR = "../data/landing"

In [2]:
customers_path = os.path.join(LANDING_DIR, "customers.csv")
order_items_path = os.path.join(LANDING_DIR, "order_items.csv")
order_payments_path = os.path.join(LANDING_DIR, "order_payments.csv")
order_reviews_path = os.path.join(LANDING_DIR, "order_reviews.csv")
orders_path = os.path.join(LANDING_DIR, "orders.csv")
products_path = os.path.join(LANDING_DIR, "products.csv")
sellers_path = os.path.join(LANDING_DIR, "sellers.csv")

In [3]:
customersDF = pl.read_csv(customers_path)
orderItemsDF = pl.read_csv(order_items_path)
orderPaymentsDF = pl.read_csv(order_payments_path)
orderReviewsDF = pl.read_csv(order_reviews_path)
ordersDF = pl.read_csv(orders_path)
productsDF = pl.read_csv(products_path)
sellersDF = pl.read_csv(sellers_path)

## Converting date and time string from string to datetime

In [4]:
orders_datetime_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
orderReviews_dateTime_cols = [
    "review_creation_date",
    "review_answer_timestamp",
]

ordersDF = ordersDF.with_columns(
    [pl.col(col).str.to_datetime() for col in orders_datetime_cols]
)
# print(ordersDF)

orderReviewsDF = orderReviewsDF.with_columns(
    [pl.col(col).str.to_datetime() for col in orderReviews_dateTime_cols]
)
# print(orderReviewsDF)

## Filling missing values

In [5]:
orders_datetime_cols = [
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
]

orderReviews_null_cols = [
    "review_comment_title",
    "review_comment_message",
]

# we do not need to fill null values in ordersDF as it is date and time columns and we cannot mess with date and time.

productsDF = productsDF.with_columns(
    pl.col("product_category_name").fill_null(pl.lit("null")),
    pl.col([
        "product_name_lenght",
        "product_description_lenght",
        "product_photos_qty",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm",
    ]).fill_null(pl.lit(0))
)

orderReviewsDF = orderReviewsDF.with_columns(
    pl.col(orderReviews_null_cols).fill_null(pl.lit("no comment"))
)

## Removing data redundancy and keeping records consistent.

In [6]:
order_payments_summary = orderPaymentsDF.group_by("order_id").agg(
    pl.col("payment_value").sum().alias("total_payment")
)

## Left Join two tables

In [8]:
fact_orders = ordersDF.join(order_payments_summary, on="order_id", how="left")
# print(fact_orders.head())

In [9]:
customers_location = customersDF.select([
    "customer_id",
    pl.concat_str(
        [pl.col("customer_city"), pl.col("customer_state")],
        separator=", "
    ).alias("customer_location")
])

In [10]:
fact_orders = fact_orders.join(customers_location, on="customer_id", how="left")

In [11]:
product_details = productsDF.select(["product_id", "product_category_name"])

In [12]:
fact_order_items = orderItemsDF.join(product_details, on="product_id", how="left")
# fact_order_items

In [13]:
STAGING_DIR = "../data/staging"
os.makedirs(STAGING_DIR, exist_ok=True)

In [14]:
fact_orders_dest = os.path.join(STAGING_DIR, "fact_orders.parquet")
fact_order_items_dest = os.path.join(STAGING_DIR, "fact_order_items.parquet")

fact_orders.write_parquet(fact_orders_dest)
fact_order_items.write_parquet(fact_order_items_dest)

In [15]:
fact_orders_path = os.path.join(STAGING_DIR, "fact_orders.parquet")
fact_order_items_path = os.path.join(STAGING_DIR, "fact_order_items.parquet")

fact_orders = pl.read_parquet(fact_orders_path)
fact_order_items = pl.read_parquet(fact_order_items_path)

In [28]:
fact_order_items_revenue = (
    fact_order_items
    .group_by("product_category_name")
    .agg(pl.col("price").sum().alias("total_category_revenue"))
    .sort("total_category_revenue", descending=True)
)
print(fact_order_items_revenue.head())

shape: (5, 2)
┌────────────────────────┬────────────────────────┐
│ product_category_name  ┆ total_category_revenue │
│ ---                    ┆ ---                    │
│ str                    ┆ f64                    │
╞════════════════════════╪════════════════════════╡
│ beleza_saude           ┆ 1.2587e6               │
│ relogios_presentes     ┆ 1.2050e6               │
│ cama_mesa_banho        ┆ 1.0370e6               │
│ esporte_lazer          ┆ 988048.97              │
│ informatica_acessorios ┆ 911954.32              │
└────────────────────────┴────────────────────────┘


In [26]:
biggest_customer_base = (
    fact_orders
    .group_by("customer_location")
    .agg(pl.len().alias("order_count"))
    .sort("order_count", descending=True)
)

print(biggest_customer_base.head())

shape: (5, 2)
┌────────────────────┬─────────────┐
│ customer_location  ┆ order_count │
│ ---                ┆ ---         │
│ str                ┆ u32         │
╞════════════════════╪═════════════╡
│ sao paulo, SP      ┆ 15540       │
│ rio de janeiro, RJ ┆ 6882        │
│ belo horizonte, MG ┆ 2773        │
│ brasilia, DF       ┆ 2131        │
│ curitiba, PR       ┆ 1521        │
└────────────────────┴─────────────┘


In [17]:
total_revenue = fact_orders.select(pl.col("total_payment").sum())
# print("Total Revenue: $", total_revenue)